In [44]:
#INSTALL
%pip install pandas
%pip install seaborn
%pip install spacy
%pip install fr_core_news_sm
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
#IMPORT
import pandas as pd
import seaborn as sns
import spacy
from spacy.lang.fr.stop_words import STOP_WORDS as fr_stop
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [46]:
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

stop_words_cuisine = [ # Générés par Gemini
    # --- Unités et Mesures ---
    "g", "gr", "gramme", "grammes", "kg", "kilogramme",
    "l", "litres", "litre", "ml", "cl", "dl",
    "cuillère", "cuillères", "c.à.s", "c.à.c", "cas", "cac",
    "pincée", "pincées", "tasse", "tasses", "verre", "verres",
    "poignée", "tranche", "tranches", "morceau", "morceaux",
    "bouteille", "sachet", "boîte", "pot", "paquet", "brin", 
    "feuille", "feuilles", "gousse", "gousses", "°","c", "mn",
    
    # --- Verbes d'action (Instructions fréquentes) ---
    "ajouter", "ajoutez", "mélanger", "mélangez", "remuer",
    "verser", "versez", "incorporer", "couper", "coupez",
    "hacher", "éplucher", "cuire", "laisser", "mettre", "mettez",
    "servir", "disposer", "saupoudrer", "garnir", "préparer",
    "chauffer", "préchauffer", "bouillir", "mijoter", "réserver",
    "laver", "égoutter", "battre", "fouetter", "étaler", "faire",
    
    # --- Adjectifs / États ---
    "chaud", "froide", "tiède", "bouillant", 
    "fondu", "haché", "râpé", "entier", "entière",
    "fin", "fine", "épais", "gros", "petit",
    "frais", "fraîche", "sec", "sèche",
    
    # --- Ingrédients "neutres" (souvent exclus de l'analyse thématique) ---
    # À retirer de cette liste si vous voulez garder tous les ingrédients
    "sel", "poivre", "eau", "huile", "beurre", "sucre", "farine",
    
    # --- Mots de liaison temporels ---
    "puis", "ensuite", "enfin", "pendant", "environ", "minutes", "min", 
    "heure", "heures", "seconde", "secondes", "avant", "après",

    # --- Autres ---
    " ", "1/2","bien"
]
stops = list(set(list(fr_stop)+stop_words_cuisine))

In [47]:
#TF-IDF
plats_df = train_df[train_df["type"] == "Plat principal"]
desserts_df = train_df[train_df["type"] == "Dessert"]
entrees_df = train_df[train_df["type"] == "Entrée"]

plats_df["text"] = plats_df["titre"] + " " + plats_df["ingredients"] + " " + plats_df["recette"]
desserts_df["text"] = desserts_df["titre"] + " " + desserts_df["ingredients"] + " " + desserts_df["recette"]
entrees_df["text"] = entrees_df["titre"] + " " + entrees_df["ingredients"] + " " + entrees_df["recette"]

vectorizer_plat = TfidfVectorizer(stop_words=stops)
vectorizer_dessert = TfidfVectorizer(stop_words=stops)
vectorizer_entree = TfidfVectorizer(stop_words=stops)

tfidf_plats = vectorizer_plat.fit_transform(plats_df["text"])
tfidf_desserts = vectorizer_dessert.fit_transform(desserts_df["text"])
tfidf_entrees = vectorizer_entree.fit_transform(entrees_df["text"])

vocab_plat = vectorizer_plat.vocabulary_
vocab_dessert = vectorizer_dessert.vocabulary_
vocab_entree = vectorizer_entree.vocabulary_

print(entrees_df.head())

c:\Main_Doc\M1\S2\TAL\projet\tal_project\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['neuf', 'qu', 'quelqu'] not in stop_words.
  warnings.warn(


                doc_id                                     titre    type  \
1    recette_48656.xml              Cake poulet/moutarde/amandes  Entrée   
2    recette_30049.xml  Bûche à la truite fumée (7ème rencontre)  Entrée   
4   recette_217204.xml                    Crêpes au canard laqué  Entrée   
5    recette_86256.xml                     Verrines aux asperges  Entrée   
18   recette_55831.xml                           Tourte lorraine  Entrée   

               difficulte        cout  \
1             Très facile  Bon marché   
2   Moyennement difficile  Assez Cher   
4   Moyennement difficile       Moyen   
5                  Facile       Moyen   
18            Très facile  Bon marché   

                                          ingredients  \
1   - 3 œufs - 150 g de farine - 1 sachet de levur...   
2   - 800 g de filet de truite saumonnée fumée en ...   
4   - 90 g de farine - 45 g de maïzena - 2 œufs - ...   
5   - 1 sachet de 500 g d'asperges surgelées - 2 c...   
18  - 1 pât

In [48]:
nlp = spacy.load("fr_core_news_sm")
def tfidf (text) : 
    tokenized = nlp(text)
    plat_somme = 0
    dessert_somme = 0
    entree_somme = 0
    for word in tokenized :
        if word.text in vocab_plat : 
            plat_somme += vectorizer_plat.idf_[vocab_plat[word.text]]
        if word.text in vocab_dessert :
            dessert_somme += vectorizer_dessert.idf_[vocab_dessert[word.text]]
        if word.text in vocab_entree :
            entree_somme += vectorizer_entree.idf_[vocab_entree[word.text]]
    if plat_somme <= dessert_somme and plat_somme <= entree_somme :
        return "Plat principal"
    elif dessert_somme <= plat_somme and dessert_somme <= entree_somme :
        return "Dessert"
    else :
        return "Entrée"

In [49]:
#TEST

test_df["text"] = test_df["titre"] + " " + test_df["ingredients"] + " " + test_df["recette"]
gg = 0 #good guesses
for index, row in test_df.iterrows() :
    res = tfidf(row["text"])
    if res == row["type"] :
        gg += 1

print("Résultat : "  + str(gg/len(test_df)))

Résultat : 0.7089337175792507


In [50]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=stops, lowercase=True)), 
    ('clf', GaussianNB())
])

X_train = train_df["titre"] + " " + train_df["ingredients"] + " " + train_df["recette"]

model.fit(X_train, train_df["type"])

predictions = model.predict(test_df["text"])
print(f"Résultat (Accuracy) : {accuracy_score(test_df["type"], predictions):.2f}")

c:\Main_Doc\M1\S2\TAL\projet\tal_project\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['neuf', 'qu', 'quelqu'] not in stop_words.
  warnings.warn(


TypeError: Sparse data was passed for X, but dense data is required. Use '.toarray()' to convert to a dense numpy array.